# ML Project 2 - Causal Inference

## 1. Import Libraries and Load Data

In [34]:
import numpy as np
import pandas as pd

import statsmodels.api as sm

import warnings

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [35]:
df_desc = pd.read_stata("../data/oregonhie_descriptive_vars.dta")
df_state = pd.read_stata("../data/oregonhie_stateprograms_vars.dta")
df_ed = pd.read_stata("../data/oregonhie_ed_vars.dta")

# merge by person_id
df = df_desc.merge(df_state, on="person_id", how="left").merge(
    df_ed, on="person_id", how="left"
)

df.head()

,person_id,household_id,treatment,draw_treat,draw_lottery,applied_app,approved_app,dt_notify_lottery,dt_retro_coverage,dt_app_decision,...,ed_charg_tot_pre_ed,ed_charg_tot_ed,any_hiun_pre_ed,any_hiun_ed,num_hiun_pre_cens_ed,num_hiun_cens_ed,any_loun_pre_ed,any_loun_ed,num_loun_pre_cens_ed,num_loun_cens_ed
0,1.0,100001.0,Selected,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Submitted an Application to OHP,No,2008-08-12,2008-09-08,2008-12-31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,100002.0,Selected,Draw 6: selected in lottery 07/01/2008,Lottery Draw 6,Did NOT submit an application to OHP,No,2008-07-14,2008-08-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,100003.0,Not selected,NaN,Lottery Draw 2,NaN,NaN,2008-04-07,2008-04-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,100004.0,Not selected,NaN,Lottery Draw 8,NaN,NaN,2008-09-11,2008-10-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,100005.0,Selected,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Did NOT submit an application to OHP,No,2008-08-12,2008-09-08,NaT,...,0.0,0.0,No,No,0.0,0.0,No,No,0.0,0.0


## 2. Part(a) - Preprocessing + Balance checks (OLS)

In [36]:
# create number of people in household on lottery list (numhh_list) dummies
numhh_dummies = pd.get_dummies(
    df["numhh_list"], prefix="numhh", drop_first=True, dtype="int"
)
df = pd.concat([df, numhh_dummies], axis=1)

df.head()

,person_id,household_id,treatment,draw_treat,draw_lottery,applied_app,approved_app,dt_notify_lottery,dt_retro_coverage,dt_app_decision,...,any_hiun_pre_ed,any_hiun_ed,num_hiun_pre_cens_ed,num_hiun_cens_ed,any_loun_pre_ed,any_loun_ed,num_loun_pre_cens_ed,num_loun_cens_ed,numhh_signed self up + 1 additional person,numhh_signed self up + 2 additional people
0,1.0,100001.0,Selected,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Submitted an Application to OHP,No,2008-08-12,2008-09-08,2008-12-31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,2.0,100002.0,Selected,Draw 6: selected in lottery 07/01/2008,Lottery Draw 6,Did NOT submit an application to OHP,No,2008-07-14,2008-08-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,3.0,100003.0,Not selected,NaN,Lottery Draw 2,NaN,NaN,2008-04-07,2008-04-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,4.0,100004.0,Not selected,NaN,Lottery Draw 8,NaN,NaN,2008-09-11,2008-10-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,5.0,100005.0,Selected,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Did NOT submit an application to OHP,No,2008-08-12,2008-09-08,NaT,...,No,No,0.0,0.0,No,No,0.0,0.0,0,0


In [37]:
# map treatment into binary variables
df["treatment"].replace({"Selected": 1, "Not selected": 0}, inplace=True)

# handle missing value in sample_ed
df["sample_ed"] = df["sample_ed"].fillna(0)

# map female_list into binary variables
df["female_list"].replace({"1: Female": 1, "0: Male": 0}, inplace=True)

df


,person_id,household_id,treatment,draw_treat,draw_lottery,applied_app,approved_app,dt_notify_lottery,dt_retro_coverage,dt_app_decision,...,any_hiun_pre_ed,any_hiun_ed,num_hiun_pre_cens_ed,num_hiun_cens_ed,any_loun_pre_ed,any_loun_ed,num_loun_pre_cens_ed,num_loun_cens_ed,numhh_signed self up + 1 additional person,numhh_signed self up + 2 additional people
0,1.0,100001.0,1,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Submitted an Application to OHP,No,2008-08-12,2008-09-08,2008-12-31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,2.0,100002.0,1,Draw 6: selected in lottery 07/01/2008,Lottery Draw 6,Did NOT submit an application to OHP,No,2008-07-14,2008-08-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,3.0,100003.0,0,NaN,Lottery Draw 2,NaN,NaN,2008-04-07,2008-04-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,4.0,100004.0,0,NaN,Lottery Draw 8,NaN,NaN,2008-09-11,2008-10-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,5.0,100005.0,1,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Did NOT submit an application to OHP,No,2008-08-12,2008-09-08,NaT,...,No,No,0.0,0.0,No,No,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74917,74918.0,174918.0,0,NaN,Lottery Draw 6,NaN,NaN,2008-07-14,2008-08-08,NaT,...,No,Yes,0.0,4.0,No,Yes,0.0,1.0,0,0
74918,74919.0,174919.0,1,Draw 6: selected in lottery 07/01/2008,Lottery Draw 6,Submitted an Application to OHP,Yes,2008-07-14,2008-08-08,2008-10-23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
74919,74920.0,174920.0,0,NaN,Lottery Draw 6,NaN,NaN,2008-07-14,2008-08-08,NaT,...,Yes,No,1.0,0.0,No,No,0.0,0.0,0,0
74920,74921.0,174921.0,1,Draw 1: selected in lottery 03/05/2008,Lottery Draw 1,Submitted an Application to OHP,Yes,2008-03-10,2008-03-11,2008-04-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


### Full sample balance check

In [38]:
y = df["sample_ed"]

X = pd.concat([df[["treatment"]], numhh_dummies], axis=1)
X = sm.add_constant(X)

full_model = sm.OLS(endog=y, exog=X, missing="drop")
result = full_model.fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:              sample_ed   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     59.03
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           4.17e-38
Time:                        18:05:35   Log-Likelihood:                -49627.
No. Observations:               74922   AIC:                         9.926e+04
Df Residuals:                   74918   BIC:                         9.930e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

### Sample balance checks


In [39]:
# restrict to ED samples
df_ed_sample = df[df["sample_ed"] != 0.0].copy()

df_ed_sample

,person_id,household_id,treatment,draw_treat,draw_lottery,applied_app,approved_app,dt_notify_lottery,dt_retro_coverage,dt_app_decision,...,any_hiun_pre_ed,any_hiun_ed,num_hiun_pre_cens_ed,num_hiun_cens_ed,any_loun_pre_ed,any_loun_ed,num_loun_pre_cens_ed,num_loun_cens_ed,numhh_signed self up + 1 additional person,numhh_signed self up + 2 additional people
4,5.0,100005.0,1,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Did NOT submit an application to OHP,No,2008-08-12,2008-09-08,NaT,...,No,No,0.0,0.0,No,No,0.0,0.0,0,0
7,8.0,102094.0,0,NaN,Lottery Draw 8,NaN,NaN,2008-09-11,2008-10-08,NaT,...,No,Yes,0.0,2.0,No,No,0.0,0.0,1,0
8,9.0,100009.0,0,NaN,Lottery Draw 1,NaN,NaN,2008-03-10,2008-03-11,NaT,...,Yes,No,1.0,0.0,No,No,0.0,0.0,0,0
15,16.0,140688.0,0,NaN,Lottery Draw 2,NaN,NaN,2008-04-07,2008-04-08,NaT,...,Yes,Yes,1.0,5.0,No,No,0.0,0.0,1,0
17,18.0,100018.0,0,NaN,Lottery Draw 4,NaN,NaN,2008-05-09,2008-06-09,NaT,...,No,No,0.0,0.0,Yes,No,2.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74906,74907.0,174907.0,1,Draw 3: selected in lottery 04/08/2008,Lottery Draw 3,Submitted an Application to OHP,No,2008-04-16,2008-05-08,2008-07-18,...,No,No,0.0,0.0,No,No,0.0,0.0,0,0
74910,74911.0,174911.0,1,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Submitted an Application to OHP,No,2008-08-12,2008-09-08,2008-12-01,...,No,No,0.0,0.0,No,No,0.0,0.0,1,0
74914,74915.0,174915.0,1,Draw 8: selected in lottery 09/02/2008,Lottery Draw 8,Submitted an Application to OHP,Yes,2008-09-11,2008-10-08,2008-11-17,...,No,No,0.0,0.0,No,No,0.0,0.0,0,0
74917,74918.0,174918.0,0,NaN,Lottery Draw 6,NaN,NaN,2008-07-14,2008-08-08,NaT,...,No,Yes,0.0,4.0,No,Yes,0.0,1.0,0,0


In [40]:
# align index
numhh_dummies_aligned = numhh_dummies.reindex(df_ed_sample.index)

X_sample = pd.concat([df_ed_sample[["treatment"]], numhh_dummies_aligned], axis=1)
X_sample = sm.add_constant(X_sample)

X_sample

,const,treatment,numhh_signed self up + 1 additional person,numhh_signed self up + 2 additional people
4,1.0,1,0,0
7,1.0,0,1,0
8,1.0,0,0,0
15,1.0,0,1,0
17,1.0,0,0,0
...,...,...,...,...
74906,1.0,1,0,0
74910,1.0,1,1,0
74914,1.0,1,0,0
74917,1.0,0,0,0


#### (i) Year of birth

In [41]:
balance_yob = sm.OLS(endog=df_ed_sample["birthyear_list"], exog=X_sample)
result_balance_yob = balance_yob.fit()
print(result_balance_yob.summary())

                            OLS Regression Results                            
Dep. Variable:         birthyear_list   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.825
Date:                Fri, 27 Feb 2026   Prob (F-statistic):              0.140
Time:                        18:05:35   Log-Likelihood:                -96304.
No. Observations:               24646   AIC:                         1.926e+05
Df Residuals:                   24642   BIC:                         1.926e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

#### (ii) Female

In [42]:
balance_female = sm.OLS(endog=df_ed_sample["female_list"], exog=X_sample)
result_balance_female = balance_female.fit()
print(result_balance_female.summary())

                            OLS Regression Results                            
Dep. Variable:            female_list   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     15.03
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           9.01e-10
Time:                        18:05:36   Log-Likelihood:                -17757.
No. Observations:               24646   AIC:                         3.552e+04
Df Residuals:                   24642   BIC:                         3.556e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

#### (iii) Signed up self for lottery

#### (iv) Any ED visit, pre-randomization (censored)

#### (v) Number of ED visits, pre-randomization (censored)